[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/067.%20The%20Kitting%20%26%20Value-Added%20Service%20Scheduling%20Problem/P67-Tier-2_executed.ipynb)

---


# 67. The Kitting & Value-Added Service Scheduling Problem
## Tier 2: The WSPT-KP Greedy Heuristic

This notebook implements a priority-based heuristic for assigning kitting work to shifts. It ranks kit–shift–period operations, then greedily assigns as much of the selected kit's remaining demand as the shift's remaining capacity permits. The method is fast and transparent, but it does **not** guarantee an optimal schedule.

### Inputs and outputs

The input consists of per-kit processing times and cost savings, demand by kit and period, shift capacity and labor cost by shift and period, urgency weights, and an additional time allowance per unit. The output is a production schedule and derived measures of fulfilled demand, capacity use, labor cost, and net savings.

### Scope and model assumptions

- Production quantities are whole units, and each unit consumes its kit's processing time plus the additional time allowance.
- The additional time allowance is applied to **each unit**. This is a simplifying assumption that makes unit-level capacity accounting explicit.
- Cost savings are earned per produced unit. Labor cost is calculated from unit processing time, shift labor cost per hour, and the conversion from minutes to hours.
- Demand is tracked by kit and period; capacity and labor cost are tracked by shift and period. The example has one period.
- The priority score is a ranking rule, not an objective function optimized exactly. Inventory constraints, precedence, setup sequencing, and other operational details are outside this example's scope.

### Heuristic sequence

1. Calculate a priority score for each kit–shift–period combination with positive demand.
2. Sort operations from highest to lowest score.
3. Consider the highest-ranked operation and assign as many whole units as its remaining capacity and demand allow.
4. Update capacity and demand, then continue through the ranked operations until demand is met or no operation can use its shift's remaining capacity.

### Example data

The example has three kits and two shifts:

| Kit | Processing time (min/unit) | Cost savings ($/unit) | Demand (units) |
|---|---:|---:|---:|
| 1 | 5 | 2.50 | 60 |
| 2 | 8 | 4.00 | 40 |
| 3 | 3 | 1.80 | 80 |

Shift capacities are 480 minutes and 420 minutes. The labor costs are $15 and $18 per hour, respectively. The code below defines the full instance, including the additional time allowance and urgency weights.

## 1. Priority rule and mathematical model

A priority score combines four considerations: savings relative to processing time, demand urgency, labor cost, and the shift's share of total capacity. A higher score means that the operation is considered earlier; it does not mean that the operation is guaranteed to appear in the final schedule.

For kit $k$, shift $s$, and period $t$, the score used in the implementation is

$$
P_{kst}
=
\left(\frac{v_k}{p_k+u}\right)
w_t
\left(\frac{L_{\max}}{L_{st}}\right)
\left(\frac{1}{1+C_{st}/\sum_{r}C_{rt}}\right).
$$

Here, $v_k$ is the cost saving per unit, $p_k$ is processing time per unit, and $u$ is the additional time allowance per unit. The urgency weight for period $t$ is $w_t$. The labor cost per hour is $L_{st}$, and $L_{\max}$ is the maximum labor cost in the instance. Capacity $C_{st}$ is measured in minutes; the last factor uses the share of period-$t$ capacity assigned to shift $s$. These factors implement the notebook's ranking rule: greater savings per unit time and urgency increase priority, while a higher labor cost or a larger capacity share reduces it.

Let $x_{kst}$ be the number of units of kit $k$ assigned to shift $s$ in period $t$. Feasibility requires

$$
\sum_s x_{kst} \leq d_{kt}
\qquad \forall k,t,
$$

$$
\sum_k (p_k+u)x_{kst} \leq C_{st}
\qquad \forall s,t,
$$

$$
x_{kst} \in \mathbb{Z}_{\geq 0}.
$$

The demand $d_{kt}$ is an upper bound on production, and each shift-period's total processing time cannot exceed its capacity. The integer restriction reflects that kits are produced as whole units. The heuristic constructs a feasible schedule subject to these constraints; it does not solve a separate exact optimization problem.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# A consistent plotting style keeps the schedule and diagnostic charts readable.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class KittingOperation:
    """One kit–shift–period candidate and its ranking score."""

    kit_id: int
    shift_id: int
    period_id: int
    processing_time: float
    cost_savings: float
    demand_urgency: float
    priority_score: float

    def __lt__(self, other):
        """Order operations by descending score when a priority queue is used."""
        return self.priority_score > other.priority_score


@dataclass
class HeuristicProblem:
    """Input data and ranking parameters for the kitting heuristic."""

    num_kits: int
    num_shifts: int
    num_periods: int

    # Dimensions: [kit], [kit], [kit, period], [shift, period], [shift, period].
    processing_times: np.ndarray
    cost_savings: np.ndarray
    demand: np.ndarray
    capacities: np.ndarray
    labor_costs: np.ndarray

    urgency_weights: np.ndarray = field(
        default_factory=lambda: np.array([1.0, 0.8, 0.6])
    )
    setup_time: float = 1.0  # Additional time allowance per produced unit, in minutes.


def create_heuristic_example() -> HeuristicProblem:
    """Create the three-kit, two-shift example instance."""
    return HeuristicProblem(
        num_kits=3,
        num_shifts=2,
        num_periods=1,
        processing_times=np.array([5, 8, 3]),
        cost_savings=np.array([2.50, 4.00, 1.80]),
        demand=np.array([
            [60],
            [40],
            [80],
        ]),
        capacities=np.array([
            [480],
            [420],
        ]),
        labor_costs=np.array([
            [15.0],
            [18.0],
        ]),
    )


problem = create_heuristic_example()
print(
    f"Heuristic problem created: {problem.num_kits} kits, "
    f"{problem.num_shifts} shifts, {problem.num_periods} periods"
)

## 2. Calculate and rank operation priorities

The score is calculated separately for every kit–shift–period candidate with positive demand. Ranking candidates rather than kits alone allows the same kit to have different priorities on different shifts because labor costs and capacity shares can differ. The ranking is fixed for the scheduling run: the score formula does not depend on the amount of demand remaining.

In [ ]:
def calculate_priority_score(
    kit_id: int,
    shift_id: int,
    period_id: int,
    problem: HeuristicProblem,
) -> float:
    """Calculate the WSPT-KP ranking score for one candidate operation."""
    processing_time = (
        problem.processing_times[kit_id] + problem.setup_time
    )
    cost_savings = problem.cost_savings[kit_id]
    base_ratio = cost_savings / processing_time

    # Later periods receive lower urgency when the configured weights decline.
    urgency_index = min(period_id, len(problem.urgency_weights) - 1)
    urgency_factor = problem.urgency_weights[urgency_index]

    max_labor_cost = np.max(problem.labor_costs)
    labor_adjustment = (
        max_labor_cost / problem.labor_costs[shift_id, period_id]
    )

    total_capacity = np.sum(problem.capacities[:, period_id])
    shift_capacity_ratio = (
        problem.capacities[shift_id, period_id] / total_capacity
    )
    capacity_factor = 1.0 / (1.0 + shift_capacity_ratio)

    return (
        base_ratio
        * urgency_factor
        * labor_adjustment
        * capacity_factor
    )


def generate_priority_queue(
    problem: HeuristicProblem,
) -> List[KittingOperation]:
    """Build and sort all candidate operations with positive demand."""
    operations = []

    for kit_id in range(problem.num_kits):
        for shift_id in range(problem.num_shifts):
            for period_id in range(problem.num_periods):
                if problem.demand[kit_id, period_id] <= 0:
                    continue

                urgency_index = min(
                    period_id, len(problem.urgency_weights) - 1
                )
                operations.append(
                    KittingOperation(
                        kit_id=kit_id,
                        shift_id=shift_id,
                        period_id=period_id,
                        processing_time=problem.processing_times[kit_id],
                        cost_savings=problem.cost_savings[kit_id],
                        demand_urgency=problem.urgency_weights[urgency_index],
                        priority_score=calculate_priority_score(
                            kit_id, shift_id, period_id, problem
                        ),
                    )
                )

    operations.sort(key=lambda operation: operation.priority_score, reverse=True)
    return operations


operations = generate_priority_queue(problem)

print("Priority calculations for all operations:")
print("=" * 80)
print(
    f"{'Kit':<6} {'Shift':<8} {'Period':<10} "
    f"{'Proc. time':<12} {'Cost saving':<13} {'Priority':<12}"
)
print("=" * 80)

for operation in operations:
    print(
        f"{operation.kit_id + 1:<6} "
        f"{operation.shift_id + 1:<8} "
        f"{operation.period_id + 1:<10} "
        f"{operation.processing_time:<12.1f} "
        f"${operation.cost_savings:<12.2f} "
        f"{operation.priority_score:<12.4f}"
    )

## 3. Greedy scheduling

The algorithm stores a schedule $x_{kst}$, remaining demand, and remaining capacity. At each step it examines the highest-ranked candidate. If the candidate has demand and enough capacity for at least one unit, the assigned quantity is

$$
q = \min\!\left(d^{\mathrm{rem}}_{kt},
\left\lfloor\frac{C^{\mathrm{rem}}_{st}}{p_k+u}\right\rfloor\right).
$$

Here, $d^{\mathrm{rem}}_{kt}$ and $C^{\mathrm{rem}}_{st}$ are the remaining demand and capacity; the floor ensures that only whole units are assigned. The code updates the schedule and both remaining quantities. A candidate is removed once its shift cannot fit another unit, and candidates whose demand has been met are skipped. The ranked list is not resorted because the priority rule is static during this run.

For $N$ candidate operations, sorting takes $O(N\log N)$. The implementation uses a Python list and removes candidates from its front, which can require shifting list elements; consequently, list handling can take $O(N^2)$ time in the worst case. The schedule and candidate list require $O(N)$-scale storage, in addition to the input arrays. The iteration limit is a safety guard, not a proof of optimality.

In [ ]:
def wspt_kp_heuristic(problem: HeuristicProblem) -> Dict:
    """Construct a schedule by greedily processing ranked operations."""
    start_time = time.time()

    production_schedule = np.zeros(
        (problem.num_kits, problem.num_shifts, problem.num_periods),
        dtype=int,
    )
    remaining_capacity = problem.capacities.copy().astype(float)
    remaining_demand = problem.demand.copy().astype(int)
    operations = generate_priority_queue(problem)

    total_iterations = 0
    while operations and total_iterations < 10000:
        operation = operations[0]
        kit_id = operation.kit_id
        shift_id = operation.shift_id
        period_id = operation.period_id

        if remaining_demand[kit_id, period_id] <= 0:
            operations.pop(0)
            continue

        unit_time = (
            problem.processing_times[kit_id] + problem.setup_time
        )
        available_capacity = remaining_capacity[shift_id, period_id]

        if available_capacity >= unit_time:
            max_possible = min(
                remaining_demand[kit_id, period_id],
                int(available_capacity / unit_time),
            )

            if max_possible > 0:
                production_schedule[kit_id, shift_id, period_id] += max_possible
                remaining_capacity[shift_id, period_id] -= (
                    max_possible * unit_time
                )
                remaining_demand[kit_id, period_id] -= max_possible

        total_iterations += 1

        # Removing an unusable candidate lets the next ranked operation be considered.
        if remaining_capacity[shift_id, period_id] < unit_time:
            operations.pop(0)

    computation_time = time.time() - start_time
    total_production = int(np.sum(production_schedule))

    total_cost_savings = 0.0
    total_labor_cost = 0.0
    for kit_id in range(problem.num_kits):
        for shift_id in range(problem.num_shifts):
            for period_id in range(problem.num_periods):
                quantity = production_schedule[kit_id, shift_id, period_id]
                if quantity > 0:
                    unit_time = (
                        problem.processing_times[kit_id] + problem.setup_time
                    )
                    total_cost_savings += (
                        quantity * problem.cost_savings[kit_id]
                    )
                    total_labor_cost += (
                        quantity
                        * unit_time
                        * problem.labor_costs[shift_id, period_id]
                        / 60
                    )

    total_demand = int(np.sum(problem.demand))
    service_level = (
        100 * total_production / total_demand if total_demand > 0 else 0.0
    )
    total_capacity = float(np.sum(problem.capacities))
    used_capacity = total_capacity - float(np.sum(remaining_capacity))
    capacity_utilization = (
        100 * used_capacity / total_capacity
        if total_capacity > 0
        else 0.0
    )

    # These checks connect the reported metrics to the schedule's constraints.
    assert np.all(production_schedule.sum(axis=1) <= problem.demand)
    assert np.all(remaining_capacity >= -1e-9)
    assert np.isclose(
        total_production + np.sum(remaining_demand), total_demand
    )

    return {
        "production_schedule": production_schedule,
        "remaining_capacity": remaining_capacity,
        "remaining_demand": remaining_demand,
        "computation_time": computation_time,
        "total_production": total_production,
        "total_cost_savings": total_cost_savings,
        "total_labor_cost": total_labor_cost,
        "net_savings": total_cost_savings - total_labor_cost,
        "service_level": service_level,
        "capacity_utilization": capacity_utilization,
        "iterations": total_iterations,
    }


results = wspt_kp_heuristic(problem)

print(f"Heuristic completed in {results['computation_time']:.4f} seconds")
print(f"Total iterations: {results['iterations']}")
print(f"Service level: {results['service_level']:.1f}%")
print(f"Capacity utilization: {results['capacity_utilization']:.1f}%")
print(f"Net cost savings: ${results['net_savings']:.2f}")

## 4. Interpret the schedule and performance measures

The schedule reports produced units by kit, shift, and period. Service level is total production divided by total demand. Capacity utilization is total used capacity divided by total available capacity; it indicates how fully the modeled time resource is used, not how profitable the schedule is. Net savings equals per-unit cost savings earned minus labor cost. Unmet demand identifies which demand the greedy ranking and available capacities leave unfilled.

In [ ]:
def analyze_heuristic_results(
    results: Dict,
    problem: HeuristicProblem,
) -> None:
    """Print the schedule, service, financial, and capacity details."""
    print("=" * 70)
    print("WSPT-KP HEURISTIC SOLUTION ANALYSIS")
    print("=" * 70)

    print("\n1. PRODUCTION SCHEDULE:")
    print("-" * 40)
    for kit_id in range(problem.num_kits):
        print(
            f"\nKit {kit_id + 1} (Processing: "
            f"{problem.processing_times[kit_id]} min/unit, "
            f"Savings: ${problem.cost_savings[kit_id]:.2f}/unit):"
        )
        for shift_id in range(problem.num_shifts):
            for period_id in range(problem.num_periods):
                quantity = results["production_schedule"][
                    kit_id, shift_id, period_id
                ]
                if quantity > 0:
                    print(
                        f"  Shift {shift_id + 1}, Period {period_id + 1}: "
                        f"{quantity:.0f} units"
                    )

        kit_production = int(np.sum(results["production_schedule"][kit_id]))
        kit_demand = int(np.sum(problem.demand[kit_id]))
        fulfillment = (
            100 * kit_production / kit_demand if kit_demand > 0 else 0.0
        )
        print(
            f"  Total: {kit_production}/{kit_demand} units "
            f"({fulfillment:.1f}% fulfillment)"
        )

    print("\n2. PERFORMANCE METRICS:")
    print("-" * 30)
    print(f"Total production: {results['total_production']:.0f} units")
    print(f"Total demand: {np.sum(problem.demand):.0f} units")
    print(f"Service level: {results['service_level']:.1f}%")
    print(f"Capacity utilization: {results['capacity_utilization']:.1f}%")
    print(f"Computation time: {results['computation_time']:.4f} seconds")

    print("\n3. FINANCIAL ANALYSIS:")
    print("-" * 25)
    print(f"Total cost savings: ${results['total_cost_savings']:.2f}")
    print(f"Total labor cost: ${results['total_labor_cost']:.2f}")
    print(f"Net savings: ${results['net_savings']:.2f}")
    if results["total_production"] > 0:
        savings_per_unit = (
            results["net_savings"] / results["total_production"]
        )
        print(f"Net savings per produced unit: ${savings_per_unit:.2f}")
    else:
        print("Net savings per produced unit: N/A")

    print("\n4. CAPACITY ANALYSIS:")
    print("-" * 25)
    for shift_id in range(problem.num_shifts):
        for period_id in range(problem.num_periods):
            original = problem.capacities[shift_id, period_id]
            remaining = results["remaining_capacity"][shift_id, period_id]
            used = original - remaining
            utilization = 100 * used / original if original > 0 else 0.0
            print(
                f"Shift {shift_id + 1}, Period {period_id + 1}: "
                f"{used:.1f}/{original:.1f} min ({utilization:.1f}%)"
            )

    print("\n5. UNMET DEMAND:")
    print("-" * 20)
    unmet_total = int(np.sum(results["remaining_demand"]))
    if unmet_total > 0:
        for kit_id in range(problem.num_kits):
            for period_id in range(problem.num_periods):
                unmet = results["remaining_demand"][kit_id, period_id]
                if unmet > 0:
                    print(
                        f"Kit {kit_id + 1}, Period {period_id + 1}: "
                        f"{unmet:.0f} units"
                    )
    else:
        print("None; all demand is satisfied.")


analyze_heuristic_results(results, problem)

## 5. Visualize the schedule and diagnostics

The charts show production by kit and shift, the initial priority ranking alongside realized production, capacity utilization, and the numerical performance measures. These plots describe this heuristic's output; they are not an optimality comparison. The performance panel keeps the measures in their original units rather than combining unlike quantities into a single score.

In [ ]:
def create_heuristic_visualization(
    results: Dict,
    problem: HeuristicProblem,
):
    """Plot the schedule and the heuristic's main diagnostic measures."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "WSPT-KP Heuristic: Kitting Scheduling Results",
        fontsize=16,
        fontweight="bold",
    )

    # A kit-by-shift view makes the allocation pattern visible at a glance.
    production_data = []
    for kit_id in range(problem.num_kits):
        for shift_id in range(problem.num_shifts):
            quantity = int(
                np.sum(results["production_schedule"][kit_id, shift_id, :])
            )
            production_data.append(
                {
                    "Kit": f"Kit {kit_id + 1}",
                    "Shift": f"Shift {shift_id + 1}",
                    "Quantity": quantity,
                }
            )

    production_df = pd.DataFrame(production_data)
    production_pivot = production_df.pivot(
        index="Kit", columns="Shift", values="Quantity"
    )
    production_pivot.plot(kind="bar", ax=axes[0, 0], width=0.8)
    axes[0, 0].set_title("Production by Kit and Shift")
    axes[0, 0].set_xlabel("Kit")
    axes[0, 0].set_ylabel("Production (units)")
    axes[0, 0].tick_params(axis="x", rotation=0)
    axes[0, 0].legend(title="Shift")
    axes[0, 0].grid(True, axis="y", alpha=0.3)

    # The second axis preserves the distinct units of score and production.
    ranked_operations = generate_priority_queue(problem)
    priorities = [operation.priority_score for operation in ranked_operations]
    labels = [
        f"K{operation.kit_id + 1}-S{operation.shift_id + 1}-P{operation.period_id + 1}"
        for operation in ranked_operations
    ]
    actual_production = [
        results["production_schedule"][
            operation.kit_id,
            operation.shift_id,
            operation.period_id,
        ]
        for operation in ranked_operations
    ]
    x_positions = np.arange(len(ranked_operations))
    width = 0.4
    priority_axis = axes[0, 1]
    production_axis = priority_axis.twinx()
    priority_bars = priority_axis.bar(
        x_positions - width / 2,
        priorities,
        width,
        label="Priority score",
        color="skyblue",
        alpha=0.8,
    )
    production_bars = production_axis.bar(
        x_positions + width / 2,
        actual_production,
        width,
        label="Production",
        color="lightcoral",
        alpha=0.8,
    )
    priority_axis.set_title("Initial Priority and Realized Production")
    priority_axis.set_xlabel("Kit–Shift–Period operation")
    priority_axis.set_ylabel("Priority score")
    production_axis.set_ylabel("Production (units)")
    priority_axis.set_xticks(x_positions)
    priority_axis.set_xticklabels(labels, rotation=45, ha="right")
    priority_axis.legend(
        [priority_bars, production_bars],
        ["Priority score", "Production"],
        loc="upper right",
    )
    priority_axis.grid(True, axis="y", alpha=0.3)

    utilization_labels = []
    utilizations = []
    for shift_id in range(problem.num_shifts):
        for period_id in range(problem.num_periods):
            original = problem.capacities[shift_id, period_id]
            remaining = results["remaining_capacity"][shift_id, period_id]
            used = original - remaining
            utilization = 100 * used / original if original > 0 else 0.0
            utilization_labels.append(
                f"Shift {shift_id + 1}, Period {period_id + 1}"
            )
            utilizations.append(utilization)

    utilization_bars = axes[1, 0].bar(
        utilization_labels, utilizations, color="mediumseagreen"
    )
    axes[1, 0].set_title("Capacity Utilization")
    axes[1, 0].set_ylabel("Utilization (%)")
    axes[1, 0].set_ylim(0, 105)
    axes[1, 0].tick_params(axis="x", rotation=25)
    axes[1, 0].grid(True, axis="y", alpha=0.3)
    for bar, utilization in zip(utilization_bars, utilizations):
        axes[1, 0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{utilization:.1f}%",
            ha="center",
            va="bottom",
        )

    metrics_axis = axes[1, 1]
    metrics_axis.axis("off")
    metric_lines = [
        "PERFORMANCE MEASURES",
        "",
        f"Service level: {results['service_level']:.1f}%",
        f"Capacity utilization: {results['capacity_utilization']:.1f}%",
        f"Net savings: ${results['net_savings']:.2f}",
        f"Computation time: {results['computation_time']:.6f} s",
        f"Iterations: {results['iterations']}",
    ]
    metrics_axis.text(
        0.05,
        0.95,
        "\n".join(metric_lines),
        va="top",
        fontsize=13,
        family="monospace",
    )

    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()
    return fig


fig = create_heuristic_visualization(results, problem)

## 6. Scalability experiment

The experiment varies the numbers of kits, shifts, and periods, then records candidate-operation count, runtime, service level, and capacity utilization. Its generated test instances use random data without a fixed seed, so their exact values and results can vary between runs. Runtime also depends on the computer and software environment. These measurements illustrate behavior for the sampled instances; they do not establish a general runtime guarantee or solution-quality percentage.

In [ ]:
def scalability_analysis() -> List[Dict]:
    """Measure heuristic behavior on randomly generated instance sizes."""
    print("=" * 60)
    print("SCALABILITY ANALYSIS")
    print("=" * 60)

    test_sizes = [
        (3, 2, 1),
        (5, 3, 2),
        (10, 3, 3),
        (15, 4, 3),
    ]
    results_summary = []

    for num_kits, num_shifts, num_periods in test_sizes:
        test_problem = HeuristicProblem(
            num_kits=num_kits,
            num_shifts=num_shifts,
            num_periods=num_periods,
            processing_times=np.random.uniform(2, 10, num_kits),
            cost_savings=np.random.uniform(1.5, 5.0, num_kits),
            demand=np.random.randint(20, 100, (num_kits, num_periods)),
            capacities=np.random.randint(
                300, 600, (num_shifts, num_periods)
            ),
            labor_costs=np.random.uniform(
                12, 20, (num_shifts, num_periods)
            ),
        )

        test_results = wspt_kp_heuristic(test_problem)
        total_operations = num_kits * num_shifts * num_periods
        results_summary.append(
            {
                "kits": num_kits,
                "shifts": num_shifts,
                "periods": num_periods,
                "total_operations": total_operations,
                "computation_time": test_results["computation_time"],
                "service_level": test_results["service_level"],
                "capacity_utilization": test_results["capacity_utilization"],
            }
        )

        print(
            f"Size {num_kits} × {num_shifts} × {num_periods}: "
            f"{test_results['computation_time']:.6f} s, "
            f"service {test_results['service_level']:.1f}%, "
            f"capacity {test_results['capacity_utilization']:.1f}%"
        )

    fig, (time_axis, service_axis) = plt.subplots(1, 2, figsize=(14, 6))
    operation_counts = [
        row["total_operations"] for row in results_summary
    ]
    computation_times = [
        row["computation_time"] for row in results_summary
    ]
    service_levels = [row["service_level"] for row in results_summary]

    time_axis.plot(
        operation_counts,
        computation_times,
        "bo-",
        linewidth=2,
        markersize=8,
    )
    time_axis.set_title("Runtime by Candidate-Operation Count")
    time_axis.set_xlabel("Candidate operations (kits × shifts × periods)")
    time_axis.set_ylabel("Runtime (seconds)")
    time_axis.grid(True, alpha=0.3)

    service_axis.plot(
        operation_counts,
        service_levels,
        "go-",
        linewidth=2,
        markersize=8,
    )
    service_axis.set_title("Service Level by Candidate-Operation Count")
    service_axis.set_xlabel("Candidate operations (kits × shifts × periods)")
    service_axis.set_ylabel("Service level (%)")
    service_axis.set_ylim(0, 100)
    service_axis.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return results_summary


scalability_results = scalability_analysis()

## 7. Interpretation, strengths, and limitations

This tier is useful when a transparent ranking rule and a quick feasible schedule are valuable. The method's computational work is driven by the number of candidate kit–shift–period operations, making it suitable for exploring larger instances than the small example. Actual runtime still depends on implementation details and the computing environment.

The priority score is a heuristic design choice. Its ranking can favor a particular kit or shift, and the greedy assignments can leave demand unmet even when a different allocation might serve more valuable demand or produce greater net savings. A high capacity-utilization percentage alone does not imply a high service level or high net savings. The notebook does not calculate an exact benchmark or an optimality gap, so no percentage of optimal solution quality can be inferred from these results.

The example also simplifies operations by applying the additional time allowance per unit and by excluding inventory, sequencing, and precedence constraints. In practice, these assumptions and the priority weights should be assessed against the operating context. Natural extensions include adding operational constraints or evaluating the same heuristic on larger, representative datasets; those extensions are outside the implementation here.